In [ ]:
import seaborn as sns

from fau_colors import register_cmaps
register_cmaps()

sns.set_palette("faculties_light")

# Set presentation context
sns.set_context("notebook", font_scale=1)
sns.set_style("whitegrid")
sns.set_style("ticks")
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import os 
os.chdir('../')
sns.set_context("paper")

# Table for residuals

In [ ]:
### TABLE for residual forces and moments, first cluster
models = {
    "MRI": {
        "suffix": "_MRI"
    },
    "scaled": {
        "suffix": ''
    },
    "SIPP": {
        "suffix": '-SBP-HIT_body'
    },
    "S-gen": {
        "suffix": '_SIPP_generic'
    },
    "addB": {
        "suffix": '-addB'
    },
    "addB+SIPP": {
        "suffix": '-addB-SBP-HIT_body'
    },
    "addB+S-gen": {
        "suffix": '-addB-SBP-SGEN_None'
    },
}
for model in models:
    models[model]['F'] = {'Fx': [], 'Fy': [], 'Fz': [], "F_norm": []}
    models[model]['M'] = {'Mx': [], 'My': [], 'Mz': [], "M_norm": []}

collfunc = lambda x: x.values # np.mean(np.abs(x))
pipelines = [['HIT', 'body']]#, ['InsideHumans', 'skin']] #['HIT', 'scan']]
participants = ['P01','P02','P03','P04','P05','P06']
mass_list = [72.31, 73.35, 64.49, 89.36, 87.63,44.91]
bodyheight = [1.67,1.77,1.65,1.77,1.79,1.44]
result_df = pd.DataFrame(columns=['participant', 'trial','MSK Model', 'dimension', 'value','value_rel'])
for participant,mass,height in zip(participants, mass_list, bodyheight):
    for trial in range(1,7):
        if (participant == 'P06' and trial == 5): continue
        if (participant == 'P01' and trial > 2): continue
        for model in models:
            if model in ['MRI', 'S-gen']:
                id1 = pd.read_csv(f"results/{participant}/gait/ID_results/Trial{trial}{models[model]['suffix']}.sto", sep='\t', skiprows=6)
            else:
                id1 = pd.read_csv(f"results/{participant}/gait/ID_results/Trial_{trial}{models[model]['suffix']}.sto", sep='\t', skiprows=6)
            # discard the first and last 1000 rows for filtering artifacts
            id1 = id1.iloc[1000:-1000] 

            for dim, name in zip(['pelvis_tx_force', 'pelvis_ty_force', 'pelvis_tz_force', 'pelvis_tilt_moment', 'pelvis_rotation_moment', 'pelvis_list_moment'], ['Fx', 'Fy', 'Fz', 'Mx', 'My', 'Mz']):
                if dim in id1.columns:
                    if name in ['Fx', 'Fy', 'Fz']:
                        models[model]['F'][name].append(collfunc(id1[dim])/mass/9.81*100)
                    if name in ['Mx', 'My', 'Mz']:
                        models[model]['M'][name].append(collfunc(id1[dim])/mass/9.81*100/ height)
                else:
                    raise ValueError(f"Dimension {dim} not found in {participant} Trial {trial} {model} data.")
            models[model]['F']['F_norm'].append(np.linalg.norm([id1['pelvis_tx_force'], id1['pelvis_ty_force'], id1['pelvis_tz_force']], axis=0)/mass/9.81*100)
            models[model]['M']['M_norm'].append(np.linalg.norm([id1['pelvis_tilt_moment'], id1['pelvis_list_moment'], id1['pelvis_rotation_moment']], axis=0)/mass/9.81*100/ height)
        #result_df.loc[len(result_df)] = [participant, trial, 'addB-IK', 'f_norm', np.mean(np.linalg.norm([id2[dim] for dim in ['pelvis_tx_force', 'pelvis_ty_force', 'pelvis_tz_force']], axis=0))/mass/9.81, 0]
        #result_df.loc[len(result_df)] = [participant, trial, 'addB-IK', 'm_norm', np.mean(np.linalg.norm([id2[dim] for dim in ['pelvis_tilt_moment', 'pelvis_list_moment', 'pelvis_rotation_moment']], axis=0))/mass/9.81, 0]

In [ ]:
# Write a nice latex table from the results
# HEADER:  & \multicolumn{4}{|c|}{\bf Forces} & \multicolumn{4}{|c|}{\bf Moments}\\ \hline
#   \bf Model & $\mathbf{F}_x~[\si{\newton}]$ & $\mathbf{F}_y~[\si{\newton}]$ & $\mathbf{F}_z~[\si{\newton}]$ & $||\mathbf{F}||~[\si{\newton}]$ & $\mathbf{M}_x~[\si{\newton\meter}]$ & $\mathbf{M}_y~[\si{\newton\meter}]$ & $\mathbf{M}_z~[\si{\newton\meter}]$ & $||\mathbf{M}||~[\si{\newton\meter}]$\\ \thickhline
# Write entries as $value \pm std$ for each model, highlight the best value in each column
for model in models:
    for dim in ['F', 'M']:
        for name in ['x', 'y', 'z', '_norm']:
            models[model][dim][dim+name+'value'] = np.mean(np.abs(np.concatenate(models[model][dim][dim+name])))
            models[model][dim][dim+name+'std'] = np.std(np.abs(np.concatenate(models[model][dim][dim+name])))
# Get the best values for each dimension
best_values_01 = {}
best_values_02 = {}
for dim in ['F', 'M']:
    for name in ['x', 'y', 'z', '_norm']:
        best_values_01[dim + name] = min(models[model][dim][dim+name+'value'] for model in models if not model.startswith('addB'))
        best_values_02[dim + name] = min(models[model][dim][dim+name+'value'] for model in models if model.startswith('addB'))

# Print it nicely in LaTeX format
for model in models:
    if model == 'addB':
        print("\\hline")
    print(f"{model} ", end='')
    for dim in ['F', 'M']:
        for name in ['x', 'y', 'z', '_norm']:
            value = models[model][dim][dim+name+'value']
            std = models[model][dim][dim+name+'std']
            if value == (best_values_01[dim + name] if not model.startswith('addB') else best_values_02[dim + name]):
                print(f"& $\\mathbf{{{value:.2f} \\pm {std:.1f}}}$  ", end='')
            else:
                print(f"& ${value:.2f} \pm {std:.1f}$ ", end='')
    print("\\\\")  # End of row

# Plot for metabolics

In [ ]:
### TABLE for residual forces and moments, first cluster
models = {
    "MRI": {
        "suffix": "_MRI",
        "ik_folder": lambda x: f"results/{x}/gait/addB-ignore_phys/IK"
    },
    "scaled": {
        "suffix": '',
        "ik_folder": lambda x: f"results/{x}/gait/addB-ignore_phys/IK"
    },
    "SIPP": {
        "suffix": '-SBP-HIT_body',
        "ik_folder": lambda x: f"results/{x}/gait/addB-ignore_phys/IK"
    },
    "S-gen": {
        "suffix": '_SIPP_generic',
        "ik_folder": lambda x: f"results/{x}/gait/addB-ignore_phys/IK"
    },
    "addB": {
        "suffix": '-addB',
        "ik_folder": lambda x: f"results/{x}/gait/addB-with_phys/generic/IK"
    },
    "addB+SIPP": {
        "suffix": '-addB-SBP-HIT_body',
        "ik_folder": lambda x: f"results/{x}/gait/addB-with_phys/S0{x[-1]}_HIT/IK"
    },
    "addB+S-gen": {
        "suffix": '-addB-SBP-SGEN_None',
        "ik_folder": lambda x: f"results/{x}/gait/addB-with_phys/S0{x[-1]}_SGEN/IK"
    },
}

def KIMR15(M,q):
    mq = M*q
    mq_pos = mq.copy()
    mq_pos[mq_pos < 0] = 0
    mq_neg = mq.copy()
    mq_neg[mq_neg > 0] = 0
    h_m, h_sl_s, h_sl_l, q_cc = 0.054, 0.283, 1.423, 0.004
    #h_m, h_sl_s, h_sl_l, q_cc = 0, 0, 1, 0
    return np.mean(h_m * np.max(np.abs(q)) * np.abs(M) + h_sl_s * mq_pos - h_sl_l * mq_neg + q_cc * np.max(M*q) + M*q)

for model in models:
    models[model]['F'] = {'Fx': [], 'Fy': [], 'Fz': [], "F_norm": []}
    models[model]['M'] = {'Mx': [], 'My': [], 'Mz': [], "M_norm": []}

collfunc = lambda x: x.values # np.mean(np.abs(x))
pipelines = [['HIT', 'body']]#, ['InsideHumans', 'skin']] #['HIT', 'scan']]
participants = ['P01','P02','P03','P04','P05','P06']
for model in models:
    models[model]['MEE'] = {f"Trial{trial}": {p: [] for p in participants} for trial in range(1, 7)}

mass_list = [72.31, 73.35, 64.49, 89.36, 87.63,44.91]
bodyheight = [1.67,1.77,1.65,1.77,1.79,1.44]
speeds = [1.3, 0.8, 1.3, 0.8, 1.3, 0.8] 
result_df = pd.DataFrame(columns=['participant', 'trial','MSK Model', 'dimension', 'value','value_rel'])
for participant,mass,height in zip(participants, mass_list, bodyheight):
    for trial, speed in zip(range(1,7), speeds):
        if (participant == 'P06' and trial == 5): continue
        if (participant == 'P01' and trial > 2): continue
        for model in models:
            if model in ['MRI', 'S-gen']:
                id1 = pd.read_csv(f"results/{participant}/gait/ID_results/Trial{trial}{models[model]['suffix']}.sto", sep='\t', skiprows=6)
            else:
                id1 = pd.read_csv(f"results/{participant}/gait/ID_results/Trial_{trial}{models[model]['suffix']}.sto", sep='\t', skiprows=6)

            ik = pd.read_csv(f"{models[model]['ik_folder'](participant)}/Trial{trial}_filtered.mot", sep='\t', skiprows=10)

            if len(id1) != len(ik):
                print(f"Warning: Length mismatch for {participant} Trial {trial} {model}: ID {len(id1)} vs IK {len(ik)}")
                continue
            MEE = 0
            for col in ik.columns:
                if col.startswith('pelvis') or col.startswith('time'):
                    continue
                q = np.gradient(ik[col])*100
                M = id1[col+'_moment']
                MEE += KIMR15(M, q) / mass / speed
            models[model]['MEE'][f"Trial{trial}"][participant].append(MEE)


In [ ]:
fig = plt.figure(figsize=(5.2, 4))
#fig = plt.figure(figsize=(5, 7.5))
# Create a DataFrame with the trial names, participants, models, and MEE values
result_df = pd.DataFrame(columns=['participant', 'trial', 'MSK Model', 'MEE'])
for participant, mass, height, speed in zip(participants, mass_list, bodyheight, speeds):
    for trial in range(1, 7):
        if (participant == 'P06' and trial == 5): continue
        if (participant == 'P01' and trial > 2): continue
        for model in models:
            result_df.loc[len(result_df)] = [participant, trial, model, models[model]['MEE'][f"Trial{trial}"][participant]]
result_df['MEE'] = result_df['MEE'].apply(lambda x: np.mean(x) if isinstance(x, list) else x)
#sns.boxplot(x='MSK Model', y='MEE', data=result_df, palette=sns.color_palette("faculties_light"))
ax = fig.add_subplot(211)
x = 'trial'
hue = 'MSK Model'
#hue = 'participant'
# Only show the follwing models: 
models_of_interest = ['scaled', 'SIPP', 'S-gen', 'MRI']
result_df = result_df[result_df['MSK Model'].isin(models_of_interest)]

sns.boxplot(result_df, x=x, y='MEE', hue=hue,showmeans=True,meanprops={'marker':'^','markerfacecolor':'grey','markeredgecolor':'black','markersize':'4'}, hue_order=models_of_interest)
# Y grid
# Plot the mean for every column
# Rename the xticks
plt.xticks(labels=["1.3 m/s\nlevel", "0.8 m/s\nlevel", "1.3 m/s\nincline", "0.8 m/s\nincline", "1.3 m/s\ndecline", "0.8 m/s\ndecline"], ticks=[0, 1, 2, 3, 4, 5])
plt.legend(loc='upper left', ncol=2, title='MSK Model',bbox_to_anchor=(-0.007, 1.02), labelspacing=0.3, columnspacing=0.3, handletextpad=0.3)
plt.ylabel('Metabolic Cost\n[J/kg/m]')  # set the y-label
plt.xlabel('')
ax_ = fig.add_subplot(212)
# Print mean and std for each model
for model in models:
    mean = np.mean(result_df[result_df['MSK Model'] == model]['MEE'])
    std = np.std(result_df[result_df['MSK Model'] == model]['MEE'])
    if not np.isnan(mean):
        print(f"{model}: {mean:.2f} ± {std:.2f}")

# Legend position top left with two columns

# Create a DataFrame with the trial names, participants, models, and MEE values
result_df = pd.DataFrame(columns=['participant', 'trial', 'MSK Model', 'MEE'])
for participant, mass, height, speed in zip(participants, mass_list, bodyheight, speeds):
    for trial in range(1, 7):
        if (participant == 'P06' and trial == 5): continue
        if (participant == 'P01' and trial > 2): continue
        for model in models:
            if model.startswith('addB+'):
                m = model.split('+')[0][0] +'+'+ model.split('+')[1]
            else:
                m = model
            result_df.loc[len(result_df)] = [participant, trial, m, models[model]['MEE'][f"Trial{trial}"][participant]]
result_df['MEE'] = result_df['MEE'].apply(lambda x: np.mean(x) if isinstance(x, list) else x)
#sns.boxplot(x='MSK Model', y='MEE', data=result_df, palette=sns.color_palette("faculties_light"))

x = 'trial'
hue = 'MSK Model'
#hue = 'participant'
# Only show the follwing models: 
models_of_interest = ['addB', 'a+SIPP', 'a+S-gen']
result_df = result_df[result_df['MSK Model'].isin(models_of_interest)]

sns.boxplot(result_df, x=x, y='MEE', hue=hue,showmeans=True,meanprops={'marker':'^','markerfacecolor':'grey','markeredgecolor':'black','markersize':'4'})
# Y grid
# Plot the mean for every column
# Rename the xticks
plt.xticks(labels=["1.3 m/s\nlevel", "0.8 m/s\nlevel", "1.3 m/s\nincline", "0.8 m/s\nincline", "1.3 m/s\ndecline", "0.8 m/s\ndecline"], ticks=[0, 1, 2, 3, 4, 5])
plt.legend(loc='upper left', ncol=2, title='MSK Model', bbox_to_anchor=(-0.007, 1.02), labelspacing=0.3, columnspacing=0.3, handletextpad=0.3)
plt.ylabel('Metabolic Cost\n[J/kg/m]')  # set the y-label
plt.xlabel('')
# set the legend names manually

# Print mean and std for each model
for model in np.unique(result_df['MSK Model']):
    mean = np.mean(result_df[result_df['MSK Model'] == model]['MEE'])
    std = np.std(result_df[result_df['MSK Model'] == model]['MEE'])
    if not np.isnan(mean):
        print(f"{model}: {mean:.2f} ± {std:.2f}")




plt.tight_layout()
# Write a bold a and b in the top left corner over each subplot
plt.text(-0.05, 1.05, 'b)', transform=ax_.transAxes, fontsize=12, fontweight='bold', va='top', ha='right')
plt.text(-0.05, 1.05, 'a)', transform=ax.transAxes, fontsize=12, fontweight='bold', va='top', ha='right')
plt.savefig('../../Topics/Publications/mrigait_empkins/figures/MEE_results.pdf', bbox_inches='tight')

# Plot for joint moments (practical difference)

In [ ]:
### TABLE for residual forces and moments, first cluster
trial = 1
models = {
    "MRI": {
        "suffix": "_MRI",
        "ik_folder": lambda x: f"results/{x}/gait/addB-ignore_phys/IK"
    },
    "scaled": {
        "suffix": '',
        "ik_folder": lambda x: f"results/{x}/gait/addB-ignore_phys/IK"
    },
    "SIPP": {
        "suffix": '-SBP-HIT_body',
        "ik_folder": lambda x: f"results/{x}/gait/addB-ignore_phys/IK"
    },
    "S-gen": {
        "suffix": '_SIPP_generic',
        "ik_folder": lambda x: f"results/{x}/gait/addB-ignore_phys/IK"
    },
    "addB": {
        "suffix": '-addB',
        "ik_folder": lambda x: f"results/{x}/gait/addB-with_phys/generic/IK"
    },
    "addB+SIPP": {
        "suffix": '-addB-SBP-HIT_body',
        "ik_folder": lambda x: f"results/{x}/gait/addB-with_phys/S0{x[-1]}_HIT/IK"
    },
    "addB+S-gen": {
        "suffix": '-addB-SBP-SGEN_None',
        "ik_folder": lambda x: f"results/{x}/gait/addB-with_phys/S0{x[-1]}_SGEN/IK"
    },
}
for model in models:
    models[model]['F'] = {'Fx': [], 'Fy': [], 'Fz': [], "F_norm": []}
    models[model]['M'] = {'Mx': [], 'My': [], 'Mz': [], "M_norm": []}

collfunc = lambda x: x.values # np.mean(np.abs(x))
pipelines = [['HIT', 'body']]#, ['InsideHumans', 'skin']] #['HIT', 'scan']]
participants = ['P01','P02','P03','P04','P05','P06']
for model in models:
    models[model]['MEE'] = {f"Trial{trial}": {p: {} for p in participants} for trial in range(1, 7)}

mass_list = [72.31, 73.35, 64.49, 89.36, 87.63,44.91]
bodyheight = [1.67,1.77,1.65,1.77,1.79,1.44]
speeds = [1.3, 0.8, 1.3, 0.8, 1.3, 0.8] 
result_df = pd.DataFrame(columns=['participant', 'trial','MSK Model', 'dimension', 'value','value_rel'])


### Ciómbine all the joint moments into a nice table
participant = pp
mass = mass_list[participants.index(participant)]
height = bodyheight[participants.index(participant)]
speed = speeds[trial]



joint_moments = ['hip_flexion_r_moment', 'knee_angle_l_moment', 'ankle_angle_r_moment', "lumbar_extension_moment", "hip_adduction_r_moment"]
res_dict = {p: {m:{j: [] for j in joint_moments} for m in models} for p in participants}
res_dict_std = {p: {m:{j: [] for j in joint_moments} for m in models} for p in participants}


if (participant == 'P06' and trial == 5): raise ValueError(f"Participant {participant} Trial {trial} not available.")
if (participant == 'P01' and trial > 2): raise ValueError(f"Participant {participant} Trial {trial} not available.")
for participant in participants:
    for model in models:
        if model in ['MRI', 'S-gen']:
            id1 = pd.read_csv(f"results/{participant}/gait/ID_results/Trial{trial}{models[model]['suffix']}.sto", sep='\t', skiprows=6)
        else:
            id1 = pd.read_csv(f"results/{participant}/gait/ID_results/Trial_{trial}{models[model]['suffix']}.sto", sep='\t', skiprows=6)
        #ik = pd.read_csv(f"{models[model]['ik_folder'](participant)}/Trial{trial}_filtered.mot", sep='\t', skiprows=10)


        grf_path = f"/Users/markusgambietz/PhD/01_Python_Projects/mrigait_empkins/data/{participant}/gait/Trial{trial}_filtered.mot"
        grf = pd.read_csv(grf_path, sep='\t', skiprows=5)   

        grf_y = grf['1_ground_force_vy'].to_numpy().copy()
        grf_y[grf_y<30] = 0
        grf_y[grf_y>0] = 1

        # Find all rising edges
        rising_edges = np.where(np.diff(grf_y) == 1)[0]
        # Plot the data from the first rising edge to the next falling edge

        grf_y = grf['1_ground_force_vy'].to_numpy()
        gc_list = []    
        for i in range(len(rising_edges)-1):
            start = rising_edges[i]
            end = rising_edges[i+1]
            gc_list.append((start, end))

        # plot the lengths of each gaait cycle
        mean_gc_len = np.mean([end-start for start, end in gc_list])
        std_gc_len = np.std([end-start for start, end in gc_list])
        gc_filtered = []
        for start, end in gc_list:
            if abs(end-start-mean_gc_len) < std_gc_len:
                gc_filtered.append((start, end))

        for dim in joint_moments:
            # loadID 
            # Grab all gait cycles
            gc_list = []
            for gc in gc_filtered:
                start, end = gc
                # Get the index of the first and last gait cycle
                id1_gc = id1[dim].iloc[start:end].to_numpy().copy()
                # interpolate to 100 points
                id1_gc = np.interp(np.linspace(0, len(id1_gc)-1, 100), np.arange(len(id1_gc)), id1_gc)
                gc_list.append(id1_gc)
            mean_id1 = np.mean(gc_list, axis=0)
            std_id1 = np.std(gc_list, axis=0)
            res_dict[participant][model][dim].append(mean_id1)
            res_dict_std[participant][model][dim].append(std_id1)


In [ ]:
figure = plt.figure(figsize=(7.5, 4))
pp = "P0"
#figure = plt.figure(figsize=(15,15))
model_order = ['scaled', 'SIPP', 'S-gen', 'MRI','addB', 'addB+SIPP', 'addB+S-gen']
for i, dim in enumerate(["hip_flexion_r_moment", "hip_adduction_r_moment", "ankle_angle_r_moment", "lumbar_extension_moment"]):
    reset_here = True
    ax = figure.add_subplot(2, 2, i+1)
    for model in model_order:
        mean = np.mean(res_dict[pp][model][dim])
        std = np.std(res_dict[pp][model][dim])
        if mean == 0 and std == 0:
            continue
        if model.startswith('addB+'):
            m = model.split('+')[0][0] +'+'+ model.split('+')[1]
        else:
            m = model
        if not model.startswith('addB'):
            
            plt.plot(np.linspace(0, 100, 100), res_dict[pp][model][dim][0], label=m)
            plt.fill_between(np.linspace(0, 100, 100), res_dict[pp][model][dim][0] - res_dict_std[pp][model][dim][0], res_dict[pp][model][dim][0] + res_dict_std[pp][model][dim][0], alpha=0.2)
        else: 
            if reset_here:
                # Reset the colorpropagation
                reset_here = False
                plt.gca().set_prop_cycle(None)
            plt.plot(np.linspace(0, 100, 100), res_dict[pp][model][dim][0], linestyle='--')
            #plt.fill_between(np.linspace(0, 100, 100), res_dict[pp][model][dim][0] - res_dict_std[pp][model][dim][0], res_dict[pp][model][dim][0] + res_dict_std[pp][model][dim][0], alpha=0.2)
        
    plt.title(dim.replace('_', ' ').capitalize())
    plt.xlabel('Gait Cycle [%]')
    plt.ylabel('Moment [Nm]')
    if dim == "hip_flexion_r_moment":
        plt.legend(loc='lower center', ncol=2, title='MSK Model', labelspacing=0.3, columnspacing=0.3, handletextpad=0.3)
    plt.xlim(0, 100)
    plt.grid()
    labels = {"hip_flexion_r_moment": "Hip Flexion Moment", "knee_angle_l_moment": "Contralateral Knee Flexion Moment", "ankle_angle_r_moment": "Ankle Flexion Moment", "lumbar_extension_moment": "Lumbar Extension Moment", "hip_adduction_r_moment": "Hip Adduction Moment"}
    plt.title(labels[dim])
    #plt.yscale('log')
plt.tight_layout()
plt.savefig(f'../../Topics/Publications/mrigait_empkins/figures/joint_moments_{pp}.pdf', bbox_inches='tight')

In [ ]:
plt.plot(grf_y)